# ai-detector — phát hiện giọng nói giả tiếng Việt

Notebook **tự chứa toàn bộ mã nguồn** (37 file, 56 KB nhúng sẵn) —
không cần clone repo, không cần dataset chứa code. Import lên Kaggle là chạy được.

```
REAL (giọng thật tiếng Việt)
   └── Piper · Kokoro · OmniVoice ──> FAKE
                 └── augmentation ──> WavLM ──> Classifier ──> REAL / FAKE
```

## Notebook chia làm hai phần — chạy phần A trước

| | Làm gì | Khi nào chạy |
|---|---|---|
| **PHẦN A** | tạo dataset: ingest → generate → **kiểm tra + nghe thử** → đóng gói | chạy trước, xem dataset có ổn không |
| **PHẦN B** | huấn luyện: split → augment → WavLM → classifier → đánh giá | chỉ chạy khi dataset đã ưng ý |

Phần A có công tắc **`SMOKE = True`**: chạy thử ~40 mẫu trong vài phút để xem
engine nào hoạt động, audio nghe ra sao. Ưng rồi mới đặt `SMOKE = False` chạy thật.

## Cần bật trong panel bên phải

| Mục | Đặt thành | Vì sao |
|---|---|---|
| **Accelerator** | `GPU T4 x2` hoặc `P100` | OmniVoice (voice cloning) không chạy nổi trên CPU |
| **Internet** | `On` | tải WavLM, giọng Piper/Kokoro, cài thư viện |

Rồi **Add Input → Datasets** một bộ giọng thật tiếng Việt (VIVOS, Common Voice vi…).
Pipeline tự nhận diện định dạng — không cần chỉnh gì thêm.

> Phiên Kaggle ~9 giờ rồi **xoá sạch `/kaggle/working`**. Ô cuối phần A đóng gói
> dataset thành một zip để bạn lưu ra Dataset, phiên sau train mà khỏi tạo lại.

## 0. Chuẩn bị

In [ ]:
# Toàn bộ package aidetector + configs, nén tar.gz rồi base64.
# sha256(payload) = 9955fe10183215be…
_PAYLOAD = (
    "..."
    "AAAAAAAAAAAAAAAAAAAAAADgn3wDhO0Z4ABwAwA="
)

import base64, hashlib, io, os, sys, tarfile
from pathlib import Path

_raw = base64.b64decode(_PAYLOAD)
assert hashlib.sha256(_raw).hexdigest() == "9955fe10183215be2e77c4dd946928920e44bb35ac8f959e362fb9db226c77a6", "payload hỏng khi sao chép notebook"

WORK = Path("/kaggle/working/ai-detector")
WORK.mkdir(parents=True, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(_raw), mode="r:gz") as _tf:
    try:
        _tf.extractall(WORK, filter="data")     # Python >= 3.12
    except TypeError:
        _tf.extractall(WORK)

os.chdir(WORK)
sys.path.insert(0, str(WORK))
CFG = "configs/kaggle.yaml"


# Chạy một stage của pipeline và DỪNG notebook ngay nếu nó lỗi.
# Không dùng `!python -m aidetector ...`: trong Jupyter, lệnh shell lỗi vẫn để
# notebook chạy tiếp các ô sau, nên một stage hỏng sẽ âm thầm kéo theo cả loạt lỗi
# vô nghĩa ở dưới — hoặc tệ hơn, chạy tiếp trên dữ liệu cũ còn sót lại.
def run(*args):
    import subprocess

    cmd = [sys.executable, "-m", "aidetector", *[str(a) for a in args], "-c", CFG]
    print("$ python -m aidetector " + " ".join(str(a) for a in args) + f" -c {CFG}\n")
    if subprocess.run(cmd).returncode != 0:
        raise SystemExit(f"✖ Stage {args[0]!r} thất bại — xem log ngay phía trên, "
                         f"đừng chạy tiếp các ô sau.")


print(f"Đã bung {len(_raw) / 1024:.0f} KB mã nguồn vào {WORK}")

Cài thư viện. Kaggle có sẵn torch + CUDA nên chỉ cài phần thiếu; ba engine sinh
fake cài riêng — cái nào lỗi thì bỏ qua, ô `info` ngay dưới cho biết cái nào dùng được.

In [ ]:
!pip install -q -r requirements.txt

!pip install -q piper-tts                                                 || true
!pip install -q git+https://github.com/iamdinhthuan/Kokoro-Vietnamese.git || true
!pip install -q omnivoice                                                 || true

!apt-get -qq install -y ffmpeg > /dev/null 2>&1 || true   # cần cho augment MP3/AAC

In [ ]:
run("info")

---
# PHẦN A — Tạo dataset

Mục tiêu của phần này là ra được một corpus **đạt chuẩn và cân bằng**, kiểm tra tận
tai trước khi tốn thời gian huấn luyện.

## A1. Chọn dataset thật + đặt quy mô

`SMOKE = True` chạy thử nhanh (~40 real + 40 fake, vài phút). Xem kết quả ở A4–A5,
ưng rồi đặt `SMOKE = False` và chạy lại từ A2 để làm thật.

In [ ]:
import logging
from pathlib import Path

from aidetector.ingest import detect_adapter
from aidetector.ingest.base import describe_directory

SMOKE = True        # ← True: chạy thử nhanh · False: chạy thật
RAW = None          # ← đặt tay nếu tự dò không đúng, vd "/kaggle/input/vivos"

if SMOKE:
    N_REAL, PER_SPEAKER, N_FAKE_TTS, N_FAKE_CLONE = 60, 8, 30, 15
else:
    N_REAL, PER_SPEAKER, N_FAKE_TTS, N_FAKE_CLONE = 4000, 120, 1200, 800

# Soi TỪNG dataset đang mount rồi chọn cái dùng được, thay vì lấy bừa cái đầu tiên:
# một dataset rỗng hay sai định dạng đứng đầu bảng chữ cái sẽ làm hỏng cả phiên.
logging.getLogger("aidetector.ingest").setLevel(logging.WARNING)
mounted = sorted(p for p in Path("/kaggle/input").glob("*") if p.is_dir())
if not mounted:
    raise SystemExit("Chưa add dataset nào — Add Input → Datasets ở panel bên phải.")

print("Dataset đang mount:")
usable = []
for folder in mounted:
    try:
        adapter, score, effective = detect_adapter(folder)
    except ValueError as exc:
        reason = next((l.strip() for l in str(exc).splitlines()[1:] if l.strip()),
                      "không nhận diện được")
        print(f"  ✖ {folder.name:<26} {reason}")
        continue
    where = "" if effective == folder else f" tại {effective.relative_to(folder)}/"
    print(f"  ✔ {folder.name:<26} {adapter.name} (điểm {score:.2f}){where}")
    usable.append((score, folder))

if RAW is None:
    if not usable:
        raise SystemExit(
            "Không dataset nào chứa audio đọc được. Chi tiết:\n"
            + "\n".join(f"[{p.name}]\n" + describe_directory(p) for p in mounted)
        )
    usable.sort(key=lambda pair: -pair[0])
    RAW = str(usable[0][1])

print(f"\nNguồn REAL : {RAW}")
print(f"Chế độ     : {'CHẠY THỬ' if SMOKE else 'CHẠY THẬT'}")
print(f"Quy mô     : {N_REAL} real · {N_FAKE_TTS} fake TTS · {N_FAKE_CLONE} fake cloning")

## A2. REAL — nạp giọng thật về chuẩn corpus

`ingest` tự nhận diện loại dataset (VIVOS / Common Voice / thư mục wav / real+fake
chia sẵn) rồi ép mọi file về đúng một chuẩn:

| | |
|---|---|
| Sample rate · kênh | 16 000 Hz · mono |
| Định dạng | WAV, 16-bit PCM |
| Độ dài | 3–10 giây (file dài hơn cắt thành nhiều đoạn) |
| Mức âm lượng | RMS −23 dBFS, trần peak −1 dBFS |
| Im lặng · clipping · NaN | cắt bớt · không được có · không được có |

Real và fake dùng **chung** chuỗi chuẩn hoá này, nên mô hình không thể phân biệt hai
lớp bằng định dạng hay độ to.

In [ ]:
run("ingest", RAW, "--limit", N_REAL, "--per-speaker", PER_SPEAKER)

In [ ]:
# Chặn sớm: ba điều kiện dưới đây mà không đạt thì mọi bước sau đều vô nghĩa.
from aidetector.config import Config
from aidetector.corpus.manifest import Manifest

manifest = Manifest.load(Config.load(CFG)["paths.corpus"], required=True)
n_real = len(manifest.reals)
n_speakers = len(manifest.speakers("real"))
n_text = sum(1 for r in manifest.reals if r.text)

print(f"real={n_real} · speaker={n_speakers} · có transcript={n_text}")
problems = []
if n_real < 10:
    problems.append(f"Chỉ nạp được {n_real} audio thật — kiểm tra RAW có trỏ đúng dataset không.")
if n_speakers < 3:
    problems.append(
        f"Chỉ có {n_speakers} speaker — không chia được train/val/test speaker-disjoint. "
        "Adapter có thể đang đọc sai cấu trúc thư mục.")
if n_text == 0:
    problems.append(
        "Không có transcript nào — fake sẽ phải dùng câu dự phòng và không ghép cặp "
        "được với real. Hãy dùng bộ dữ liệu có transcript (VIVOS, Common Voice).")
if problems:
    raise SystemExit("DỪNG LẠI:\n" + "\n".join(f"  • {p}" for p in problems))
print("✔ dataset thật đủ điều kiện để sinh fake")

## A3. FAKE — sinh audio giả

Mỗi audio giả sinh từ **chính transcript và speaker của một utterance thật**, nên
luôn có bản real đối chứng cùng nội dung cùng giọng — mô hình không thể phân loại
theo chủ đề câu nói hay theo danh tính người nói.

`generate` là idempotent: dừng giữa chừng rồi chạy lại chỉ sinh phần còn thiếu.

In [ ]:
# Hai engine TTS giọng cố định — nhanh, chạy được cả trên CPU.
run("generate", "--engines", "piper", "kokoro", "--count", N_FAKE_TTS)

In [ ]:
# OmniVoice: voice cloning zero-shot, clone thẳng giọng speaker thật từ một câu
# khác của họ. Chậm hơn nhiều và cần GPU — bỏ qua ô này nếu chạy CPU.
run("generate", "--engines", "omnivoice", "--count", N_FAKE_CLONE)

## A4. Kiểm tra dataset

Ba việc: soi toàn corpus xem có file nào phạm chuẩn, xem thống kê, và **nghe thử**.

In [ ]:
run("validate")

In [ ]:
# Thống kê chi tiết: số lượng, thời lượng, cân bằng hai lớp, phủ speaker
from collections import Counter

from aidetector.config import Config
from aidetector.corpus.manifest import Manifest

cfg = Config.load(CFG)
manifest = Manifest.load(cfg["paths.corpus"], required=True)
stats = manifest.stats()

n_real = stats["by_label"].get("real", 0)
n_fake = stats["by_label"].get("fake", 0)
print(f"Tổng      : {stats['total']} utt · {stats['hours']} giờ")
print(f"REAL/FAKE : {n_real} / {n_fake}"
      + (f"   ⚠ lệch {max(n_real, n_fake) / max(min(n_real, n_fake), 1):.1f}×"
         if min(n_real, n_fake) and max(n_real, n_fake) / min(n_real, n_fake) > 1.3 else "   ✔ cân bằng"))
print(f"Speaker   : {stats['speakers_real']}")

print("\nTheo engine:")
for name, count in sorted(stats["by_generator"].items()):
    print(f"  {name:<42} {count}")

durations = [r.duration for r in manifest]
print(f"\nĐộ dài    : {min(durations):.1f}–{max(durations):.1f}s "
      f"(trung bình {sum(durations) / len(durations):.1f}s)")

paired = sum(1 for r in manifest.fakes if r.ref_utt_id in manifest)
print(f"Ghép cặp  : {paired}/{len(manifest.fakes)} fake có real đối chứng cùng nội dung")

no_text = sum(1 for r in manifest.reals if not r.text)
if no_text:
    print(f"⚠ {no_text} utt real không có transcript — không dùng làm khuôn sinh fake được")

In [ ]:
# NGHE THỬ: mỗi cặp là cùng một câu, cùng một speaker — real trước, fake sau.
from IPython.display import Audio, display

pairs = []
for fake in manifest.fakes:
    real = manifest.get(fake.ref_utt_id)
    if real is not None:
        pairs.append((real, fake))
    if len(pairs) >= 3:
        break

if not pairs:
    print("Chưa có fake nào — chạy lại ô A3.")
for real, fake in pairs:
    print("=" * 90)
    print(f"Câu    : {real.text[:110]}")
    print(f"Speaker: {real.speaker}   ·   engine: {fake.generator}")
    print(f"REAL ({real.duration:.1f}s)")
    display(Audio(str(manifest.abs_path(real))))
    print(f"FAKE ({fake.duration:.1f}s)")
    display(Audio(str(manifest.abs_path(fake))))

In [ ]:
# Dạng sóng + phổ của một cặp — fake thường mượt và đều hơn ở vùng tần số cao.
import matplotlib.pyplot as plt
import numpy as np

from aidetector.corpus.spec import load_audio

if pairs:
    real, fake = pairs[0]
    fig, axes = plt.subplots(2, 2, figsize=(13, 6))
    for col, (rec, title) in enumerate([(real, "REAL"), (fake, f"FAKE · {fake.generator}")]):
        audio = load_audio(manifest.abs_path(rec), 16_000)
        axes[0, col].plot(np.arange(len(audio)) / 16_000, audio, lw=0.4)
        axes[0, col].set(title=f"{title} — dạng sóng", xlabel="giây", ylim=(-1, 1))
        axes[1, col].specgram(audio, Fs=16_000, NFFT=512, noverlap=256, cmap="magma")
        axes[1, col].set(title=f"{title} — phổ", xlabel="giây", ylabel="Hz")
    fig.tight_layout()
    plt.show()

## A5. Đóng gói dataset

`/kaggle/working` bị xoá khi hết phiên, và commit output với hàng chục nghìn file wav
rời rạc thì rất chậm — nên gói tất cả vào **một** zip.

Chạy xong notebook: **Output → New Dataset**. Phiên sau chỉ cần add dataset đó rồi
`unpack`, khỏi phải ingest và generate lại.

In [ ]:
run("pack", "--out", "/kaggle/working/corpus.zip")
!ls -lh /kaggle/working/corpus.zip

> ### Dừng lại ở đây nếu chỉ cần dataset
>
> Xem lại A4: hai lớp có cân bằng không, engine nào sinh được bao nhiêu, nghe thử
> thấy hợp lý chưa. Nếu đang ở `SMOKE = True` thì giờ đặt `SMOKE = False` ở ô A1 và
> chạy lại A2–A5 để làm thật. Ưng rồi mới sang phần B.

---
# PHẦN B — Huấn luyện

Chạy phần này khi dataset đã ưng. Nếu dataset đến từ phiên trước, chạy ô ngay dưới
để bung nó ra rồi bỏ qua toàn bộ phần A.

In [ ]:
# Chỉ chạy khi dùng lại dataset của phiên trước:
# run("unpack", "/kaggle/input/<tên-dataset>/corpus.zip")

## B1. Chia tập → augment

`split` chạy **trước** `augment`: bản augment chỉ sinh cho train và bám đúng split
của bản gốc, còn val/test giữ audio sạch để số đo phản ánh dữ liệu thật. Chia
speaker-disjoint nên không có speaker nào xuất hiện ở hai tập.

Thêm `--holdout omnivoice` nếu muốn giữ hẳn một engine riêng cho test — đó là phép
đo sát thực tế nhất: mô hình có bắt được engine **chưa từng thấy** hay không.

In [ ]:
run("split")
run("augment", "--copies", 1)

## B2. WavLM → Classifier

Embedding cache theo `utt_id` nên chạy lại chỉ trích phần mới. Đổi backbone chỉ cần
`--set features.backbone.name=wav2vec2` — cache tách riêng, không đè lên nhau.

In [ ]:
run("features")
run("train")
run("evaluate")

## B3. Kết quả

In [ ]:
import json
from pathlib import Path
from IPython.display import Image, display

metrics = json.loads(Path("/kaggle/working/reports/metrics.json").read_text())
overall = metrics["overall"]
print(f"EER      : {overall['eer'] * 100:.2f}%      ← số đo chính")
print(f"ROC-AUC  : {overall['roc_auc']:.4f}")
print(f"min-DCF  : {overall['min_dcf']:.4f}")
print(f"Accuracy : {overall['accuracy'] * 100:.2f}%  (ngưỡng {overall['threshold']:.3f})")

print("\nTheo từng generator:")
for name, entry in metrics["by_generator"].items():
    if "eer_vs_all_real" in entry:
        print(f"  {name:<42} n={entry['n']:>5} · EER {entry['eer_vs_all_real'] * 100:6.2f}%"
              f" · bắt được {entry['detection_rate'] * 100:5.1f}%")
    elif "false_alarm_rate" in entry:
        print(f"  {name:<42} n={entry['n']:>5} · báo nhầm {entry['false_alarm_rate'] * 100:5.1f}%")

print("\nClean vs augmented:")
for name, entry in metrics["by_condition"].items():
    print(f"  {name:<12} n={entry['n']:>5} · điểm trung bình {entry['mean_score']:.3f}")

display(Image("/kaggle/working/reports/curves.png"))
display(Image("/kaggle/working/reports/confusion_matrix.png"))

## B4. Thử trên file bất kỳ + lưu mô hình

In [ ]:
import glob

mau = sorted(glob.glob("/kaggle/working/corpus/audio/fake/piper/*/*.wav"))[:5]
mau += sorted(glob.glob("/kaggle/working/corpus/audio/real/*/*/*.wav"))[:5]
run("detect", *mau)

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/model",          "zip", "/kaggle/working/checkpoints")
shutil.make_archive("/kaggle/working/reports_bundle", "zip", "/kaggle/working/reports")
!ls -lh /kaggle/working/*.zip

---
### Vài nút chỉnh hay dùng

```python
# Đổi backbone (cache đặc trưng tách riêng nên không đụng nhau)
run("run", "features", "train", "evaluate", "--set", "features.backbone.name=wav2vec2")

# Đo khả năng tổng quát sang engine chưa từng thấy
run("split", "--holdout", "omnivoice")
run("run", "features", "train", "evaluate")

# Augment mạnh tay hơn nếu clean và augmented chênh lệch nhiều
run("augment", "--copies", 3, "--set", "augment.ops.codec.p=0.8")
```

Toàn bộ tham số nằm trong `configs/default.yaml` (bản Kaggle kế thừa nó qua
`configs/kaggle.yaml`) — xem bằng `!cat configs/default.yaml`.